# MLFLOW first look

In [ ]:
#!conda install -c conda-forge mlflow -y

In [ ]:
import sys
#!{sys.executable} -m pip install --upgrade pip setuptools wheel
#!{sys.executable} -m pip install mlflow scikit-learn pandas matplotlib

In [8]:
# trying differents version of it
#!{sys.executable} -m pip install --force-reinstall mlflow==2.9.2 scikit-learn==1.3.2 pandas matplotlib numpy

In [7]:
import mlflow
print("✅ MLflow version:", mlflow.__version__)

✅ MLflow version: 3.1.4


# Loading data and libraries

In [14]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [15]:
# Load data
db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# First Simple Example

In [16]:
# set the experiment id
mlflow.set_experiment(experiment_id="0")

mlflow.autolog()
db = load_diabetes()

X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=250, max_depth=7, max_features=5)
rf.fit(X_train, y_train)

# Use the model to make predictions on the test dataset.
predictions = rf.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print("📊 Test MSE:", mse)
print("📊 Test R²:", r2)

2025/10/18 13:58:34 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/10/18 13:58:34 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2bf71edfe3df4fe2854d8283e488c2c8', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


📊 Test MSE: 3371.146623708313
📊 Test R²: 0.4460957448228957


# Finding the best model

In [20]:
# Set experiment name (not ID)
from sklearn.metrics import mean_squared_error, r2_score

# Set experiment name
mlflow.set_experiment("diabetes_rf_search")
mlflow.autolog()

# Define parameter grid
param_grid = {
    "n_estimators": [50, 250,100],
    "max_depth": [4, 6,7,8],
    "max_features": [2, 3,4,5]
}

# Grid search
search = GridSearchCV(
    RandomForestRegressor(),
    param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    verbose=0
)

search.fit(X_train, y_train)

# Log test metrics manually
with mlflow.start_run(run_name="best_model_test_metrics"):
    best_model = search.best_estimator_
    predictions = best_model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    mlflow.log_params(search.best_params_)
    mlflow.log_metric("test_mse", mse)
    mlflow.log_metric("test_r2", r2)
    mlflow.sklearn.log_model(best_model, name="best_rf_model")

    print("✅ Best parameters:", search.best_params_)
    print("📊 Test MSE:", mse)
    print("📊 Test R²:", r2)

2025/10/18 14:00:39 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/10/18 14:00:39 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2d54f520f9564104b8ab6f29403de0fa', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/10/18 14:01:07 INFO mlflow.sklearn.utils: Logging the 5 best runs, 43 runs will be omitted.
2025/10/18 14:01:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Best parameters: {'max_depth': 4, 'max_features': 4, 'n_estimators': 100}
📊 Test MSE: 3281.766117083688
📊 Test R²: 0.4607816213140242


We find a better parameter for the model thanks to mlflow and GridSearchCV working together. We can see some additiona information here: http://127.0.0.1:5000/

# Selecting different models
selecting the best of them

In [48]:
# selecting of the run the best model
run_id = "f514f553e7864fb3887b6cbb9d34ceef"  # paste the actual ID here
model_uri = f"runs:/{run_id}/best_rf_model"

best_model = mlflow.sklearn.load_model(model_uri)

# Use it for predictions
predictions = second_best_model.predict(X_test)

In [49]:
best_model

RandomForestRegressor(max_depth=4, max_features=4)

Find it another one of them

In [41]:
from mlflow.tracking import MlflowClient

In [43]:
client = MlflowClient()
experiment = client.get_experiment_by_name("diabetes_rf_search")
runs = client.search_runs(experiment.experiment_id, order_by=["metrics.test_mse ASC"], max_results=1000)

# Convert to DataFrame for easy filtering
df = pd.DataFrame([{
    "run_id": r.info.run_id,
    "mse": r.data.metrics.get("test_mse"),
    "r2": r.data.metrics.get("test_r2"),
    **r.data.params
} for r in runs])

# Drop runs without test metrics
df = df.dropna(subset=["mse"])

# Sort by performance
df_sorted = df.sort_values("mse")
df_sorted.head()

,run_id,mse,r2,max_depth,max_features,n_estimators,bootstrap,ccp_alpha,criterion,max_leaf_nodes,...,best_max_features,best_n_estimators,cv,error_score,estimator,param_grid,pre_dispatch,refit,return_train_score,scoring
0,f514f553e7864fb3887b6cbb9d34ceef,3281.766117,0.460782,4,4,100,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,08c0e7f2840841c987e39239ea148f02,3354.336920,0.448858,6,3,100,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2e22ae8d975a4ee782e1912e5b61b245,3404.793828,0.440567,7,5,250,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
# selecting of the run another model
run_id = "08c0e7f2840841c987e39239ea148f02"  # paste the actual ID here
model_uri = f"runs:/{run_id}/best_rf_model"

second_best_model = mlflow.sklearn.load_model(model_uri)

# Use it for predictions
predictions = second_best_model.predict(X_test)

In [50]:
second_best_model

RandomForestRegressor(max_depth=6, max_features=3)